# 04 · Satellite features from Google Earth Engine

The challenger models in notebook 05 predict deprivation from satellite data alone. This notebook exports
two sets of features for every rural locality, averaged within 500 m of its census coordinates: Google
DeepMind's AlphaEarth Foundations embeddings (64 dimensions at 10 m resolution, 2020) and VIIRS night-time
lights for the same year, a standard baseline in poverty mapping.

**Runtime:** about 15 minutes of setup, then 30-90 minutes of processing on Earth Engine. **Requires:** a
Google Cloud project registered for Earth Engine (free for noncommercial use). The notebook runs in
JupyterLab; Google Colab is a convenient alternative for this step because Earth Engine sign-in and Google
Drive are built in, and section 4 copies the exported files from Drive automatically.

In [ ]:
# Make the project importable; works in JupyterLab, VS Code and Colab without reinstalling.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from mapaudit import notebook as nb
from mapaudit import pipeline, report
from mapaudit.config import get_paths

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)
MODE = nb.notebook_mode()  # "real" by default; CI sets MAPAUDIT_NOTEBOOK_MODE=demo (synthetic data)
print("Mode:", MODE)

## 1. Sign in to Earth Engine

In [ ]:
PROJECT = "your-gcp-project-id"  # the Google Cloud project registered for Earth Engine
START_EXPORT = False  # set to True only after the dry run below succeeds

if MODE == "real":
    get_ipython().run_line_magic("pip", "install -q earthengine-api")
    import ee

    ee.Authenticate()  # opens a browser the first time; later runs reuse the saved credentials

## 2. Dry run: compute the features for one locality

In [ ]:
from mapaudit.gee import export_embeddings

if MODE == "real":
    export_embeddings(PROJECT, dry_run=True)

## 3. Start the export (about 37 tasks)

In [ ]:
if MODE == "real" and START_EXPORT:
    tasks = export_embeddings(PROJECT)

Export progress is visible at https://code.earthengine.google.com/tasks. The CSV files are written to the
Google Drive folder `mapaudit_alphaearth`.

## 4. Bring the files into `data/raw/`

- **JupyterLab:** download the CSVs from Google Drive into `data/raw/`.
- **Colab:** the next cell mounts Google Drive and copies them automatically.

In [ ]:
import shutil

if MODE == "real" and "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    for f in sorted(Path("/content/drive/MyDrive/mapaudit_alphaearth").glob("alphaearth_*_part*.csv")):
        shutil.copy2(f, get_paths("real").raw / f.name)

if MODE == "demo":
    nb.prepare_demo(through="build")
files = sorted(get_paths(MODE).raw.glob("alphaearth_*_part*.csv"))
print(f"{len(files)} embedding files in data/raw")

## 5. Rebuild with the features

The locality table is rebuilt with the features attached, and the sanity checks now include the median
length (norm) of the embedding vectors.

In [ ]:
summary = pipeline.build(MODE)
display(nb.sanity_checks(summary, MODE))
pd.Series({
    "rural localities": summary.get("localities_in_scope"),
    "with satellite features": summary.get("localities_with_embeddings"),
    "median embedding norm": summary.get("embedding_median_norm"),
}, name="value", dtype=object).to_frame()